# Training YOLO11 on custom dataset

In [ ]:
!nvidia-smi

## All imports for Training

In [ ]:
import os
from pathlib import Path

import albumentations as A
import ultralytics
from roboflow import Roboflow
from ultralytics import YOLO, settings

HOME = Path(os.getcwd())
print(HOME)

## Install YOLO11 via Ultralytics

In [ ]:
settings.reset()

ultralytics.checks()

In [ ]:
ultralytics.settings["tensorboard"] = True
ultralytics.settings["weights_dir"]     = str(HOME / "weights")
ultralytics.settings["runs_dir"]        = str(HOME / "runs")
ultralytics.settings["datasets_dir"]    = str(HOME / "datasets")
ultralytics.settings

## Download Dataset

In [ ]:
os.makedirs(HOME/"datasets", exist_ok=True)

%cd {HOME}\datasets

rf = Roboflow(api_key="")
project = rf.workspace("").project("")
version = project.version(1)
dataset = version.download("yolo11")            

print(dataset.location)

%cd {HOME}

## Training

> Run this in cli after running `model.train()`
>
> `tensorboard --logdir runs/`

In [ ]:
RUN_TYPE_NUMBER = "detect"
PRETRAINED_MODEL_NAME = "yolo11n.pt"

In [ ]:
def zero_angle_probability(p: float):
    return (4 * p) / 5

# Load the last saved model
model = YOLO(PRETRAINED_MODEL_NAME)  # or use your custom path

results = model.train(
    data=f"{dataset.location}/data.yaml",
    imgsz=640,
    epochs=500,
    batch=16,
    plots=True,
    save=True,
    save_period=10,
    patience=30,
    cache="disk",
    # lr0=1e-5,
    seed=42,

    # Augmentation
    mosaic=0.0, # Augemntation off so that model detect whole object instead partially
    erasing=0.0,
    fliplr=0.0,
    scale=0.0,
    translate=0.0,
    augmentations=[
        A.RandomRotate90(p=zero_angle_probability(p=0.5))
    ]
    
)


**NOTE:** The results of the completed training are saved in `{HOME}/runs/detect/train/`. Let's examine them.

In [ ]:
from IPython.display import Image as IPyImage

IPyImage(filename=f'{HOME}/runs/{RUN_TYPE_NUMBER}/train/confusion_matrix.png')

### Validate fine-tuned model

BEST PT

In [ ]:
import polars as pl

# Load the training log
results = pl.read_csv(f"runs/{RUN_TYPE_NUMBER}/train/results.csv")

# Strip spaces from column names
results = results.rename({
    col: col.strip()
    for col in results.columns
})

# Calculate fitness
results = results.with_columns(
    (
        pl.col("metrics/mAP50(B)") * 0.1
        + pl.col("metrics/mAP50-95(B)") * 0.9
    ).alias("fitness")
)

# Find the epoch with the highest fitness
best_epoch = results["fitness"].arg_max() + 1

print(f"Best model was saved at epoch: {best_epoch}")

## Testing

In [ ]:
best_model = YOLO(f"runs/{RUN_TYPE_NUMBER}/train/weights/best.pt")

### Testing on valid

In [ ]:
directory = f"{dataset.location}/valid/images"
pred_val_dir = "val"


val_results = best_model.predict(source=directory, save=True, project=pred_val_dir, verbose=False, iou=0.1, conf=0.1, stream=True)

for result in val_results:
    pass

### Testing on Test Case Recursively

In [ ]:
directory = Path("")

pred_val_dir = "test-outputs"

# Find all images recursively
image_extensions = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp",
    ".tif",
    ".tiff",
    ".heic",
    ".heif",
    ".mpo",
    ".dng",
    ".avif",
    ".jp2",
}

images = [
    str(p)
    for p in directory.rglob("*")
    if p.is_file() and p.suffix.lower() in image_extensions
]

print(f"Found {len(images)} images")

test_results = best_model.predict(source=images, save=True, project=pred_val_dir, verbose=False, iou=0.1, conf=0.1, stream=True)

for result in test_results:
    pass